In [1]:
# Librerie e moduli
import os 
import pandas as pd
import pigeonXT as pixt
from pigeonXT import annotate
from IPython.display import display, Image 
import random
import numpy as np
from pathlib import Path

In [2]:
# Dataset principale
df_master = pd.read_csv('bureau_data_classified_ALL.csv', delimiter='\t', usecols=["macro_area", "area", "occupation_type", "occupation", "img_name", "path", "num_people", "gender"])
# Dataset filtrato (immagini con 1 sola persona)
df_master_filtered = df_master[df_master['num_people'] == 1]

# Funzione per mostrare le immagini sul notebook
def display_fn(x):
    value = x.value
    value = value.replace("<style>div.output_scroll { height: 88em; width: 100%}</style>", "").replace("<body><div>", "").replace("</div></body>", "")
    value = display(Image(value, width=300))
    return x

# Variabili
image_directory = 'DATA/Outputs'
image_paths = []
unique_subfolders = df_master_filtered['path'].unique() # per eliminare i duplicati
image_limit_per_subfolder = 30

# Ciclo per campionare 30 immagini casuali da ciascuna sottocartella della directory principale 
for subfolder in unique_subfolders:
    # Filtro solo le righe corrispondenti alla sottocartella attuale
    filtered_subset = df_master_filtered[df_master_filtered['path'] == subfolder]
    # Campiono 30 immagini casuali da questa sottocartella
    sample = filtered_subset.sample(min(image_limit_per_subfolder, len(filtered_subset)))
    # Aggiungo i path delle immagini campionate alla lista
    for _, row in sample.iterrows():
        image_path = os.path.join(image_directory, row['path'], row['img_name'])
        image_paths.append(image_path)

# Dataset preparatorio per l'annotazione (3 colonne: path immagini + etnia + genere)
df_path_etnia = pd.DataFrame({'Path': image_paths, 'Ethnicity': None, 'Gender': None})
df_path_etnia.to_csv("df_ethn_gend.csv", index=False, sep='\t')

In [3]:
# Opzioni di annotazione (label)
labels = ['caucasian', 'african', 'asian', 'middle eastern', 'hispanic', 'male', 'female']
# File che conterrà l'output dell'annotazione (3 colonne: path immagini + annotazione etnia + annotazione genere)
outputFile = "out_ann.csv"

# Funzione per creare un array binario per rappresentare le label
def setLabels(labels_indices, numClasses):
    row = np.zeros([numClasses], dtype=np.uint8)
    row[labels_indices] = 1
    return row

# Funzione per annotare una porzione di 150 immagini alla volta
def labelPortion(inputFile, labels, outputFile, portionSize=150, textColumn="Path"):
    outputFile = Path(outputFile)
    if outputFile.exists():
        outdf = pd.read_csv(outputFile, delimiter='\t')
        currentId = outdf.index.max() + 1
    else:
        currentId = 0
   
    indf = pd.read_csv(inputFile, delimiter='\t')
    tot_imgs = len(indf)
    indf = indf.loc[currentId:currentId + portionSize - 1]
    actualPortionSize = len(indf)
    print(f'{currentId + 1} - {currentId + actualPortionSize} of {tot_imgs}')
    image_paths = indf[textColumn].tolist()
    
    indf['Ethnicity'] = None
    indf['Gender'] = None
    
    # Funzione per registrare la label scelta per ciascuna immagine
    def updateRow(img_path, selectedLabels):
        if selectedLabels:
            selected_label_ethn = selectedLabels[0]
            selected_label_gender = selectedLabels[1]
            indf.loc[indf[textColumn] == img_path, 'Ethnicity'] = selected_label_ethn
            indf.loc[indf[textColumn] == img_path, 'Gender'] = selected_label_gender
    
    # Funzione per gestire il salvataggio delle annotazioni nel file di output
    def finalProcessing(annotations):
        nonlocal outputFile
        if outputFile.exists():
            prevdata = pd.read_csv(outputFile, delimiter='\t')
            outdata = pd.concat([prevdata, indf[[textColumn, 'Ethnicity', 'Gender']]]).reset_index(drop=True)
        else:
            outdata = indf[[textColumn, 'Ethnicity', 'Gender']].copy()
        outdata.to_csv(outputFile, index=False, sep='\t')
    
    # Annotazione con PigeonXT
    annotated = pixt.annotate( 
        image_paths, 
        options=labels, 
        task_type='multilabel-classification',
        buttons_in_a_row=3,
        reset_buttons_after_click=True,
        include_next=True,
        display_fn=display_fn,
        example_process_fn=updateRow,
        final_process_fn=finalProcessing
    )
    
    return indf

In [7]:
annotations = labelPortion("df_ethn_gend.csv", labels, outputFile)

5875 - 5874 of 5874


HTML(value='0 of 0 Examples annotated, Current Position: 0 ')

Output()

Annotation done.
